<a href="https://colab.research.google.com/github/NyamburaWanjohi/Bioinformatics_project/blob/main/01_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q GEOparse

import GEOparse
import pandas as pd
import os

os.makedirs("data", exist_ok=True)

# 1. Download GSE15852 directly from NCBI GEO
gse = GEOparse.get_GEO(geo="GSE15852", destdir="./data", silent=True)

# 2. Build Expression Matrix (Rows = Probe/Gene IDs, Columns = Sample GSM IDs)
expr_matrix = gse.pivot_samples('VALUE')

# Map Affymetrix Probe IDs to official Gene Symbols using platform annotation
gpl = list(gse.gpls.values())[0]
probe_to_gene = gpl.table[['ID', 'Gene Symbol']].dropna()
probe_to_gene = probe_to_gene[probe_to_gene['Gene Symbol'] != '']
# Keep first symbol if multiple are listed (e.g., "BRCA1 /// NBR2" -> "BRCA1")
probe_to_gene['Gene_Symbol'] = probe_to_gene['Gene Symbol'].apply(lambda x: str(x).split('///')[0].strip())

expr_with_gene = expr_matrix.merge(probe_to_gene[['ID', 'Gene_Symbol']], left_index=True, right_on='ID')
expr_with_gene = expr_with_gene.drop(columns=['ID']).set_index('Gene_Symbol')

# Save expression matrix
expr_with_gene.to_csv("data/expression_matrix.csv")

# 3. Build Sample Metadata Table (metadata.csv)
meta_records = []
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata.get('title', [''])[0]
    source = gsm.metadata.get('source_name_ch1', [''])[0]
    chars = " | ".join(gsm.metadata.get('characteristics_ch1', []))

    # Determine Tumor vs Normal label
    condition = "Normal" if "normal" in (title + source + chars).lower() else "Tumor"
    # Extract patient pairing ID (e.g., patient number from title)
    patient_id = "".join([c for c in title if c.isdigit()])

    meta_records.append({
        "sample_id": gsm_name,
        "sample_title": title,
        "patient_id": f"Patient_{patient_id}" if patient_id else gsm_name,
        "condition": condition,
        "label": 1 if condition == "Tumor" else 0,
        "source_tissue": source
    })

metadata_df = pd.DataFrame(meta_records)
metadata_df.to_csv("data/metadata.csv", index=False)

# 4. Print Basic Sample-Count Table (Required Evidence)
print("Expression Matrix Shape (Genes x Samples):", expr_with_gene.shape)
print("\nSample-Count Table:")
print(metadata_df['condition'].value_counts().rename_axis('Tissue Type').reset_index(name='Sample Count'))
print("\nUnique Patients:", metadata_df['patient_id'].nunique())

Expression Matrix Shape (Genes x Samples): (21225, 86)

Sample-Count Table:
  Tissue Type  Sample Count
0      Normal            43
1       Tumor            43

Unique Patients: 43
